# Arez — Devil's Advocate
**Model:** Mistral Small 3.1 24B · **Hardware:** A100 40GB

| Bước | Cell | Làm gì |
|---|---|---|
| 1 | **Cell 1** | Cài vllm → **Restart Runtime ngay** |
| 2 | **Cell 2** | Load model (~3 phút) |
| 3 | **Cell 3** | Sửa `USER_INPUT` → Run mỗi lần hỏi |
| 4 | **Cell 2b** | Reset lịch sử |

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — CÀI VLLM
# vllm tự handle Mistral3 — không cần transformers nightly
# ⚠️ SAU KHI XONG: Runtime > Restart runtime
# ═══════════════════════════════════════════════════

import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', 'vllm'
])

print('✅ Cài vllm xong.')
print('⚠️  Bây giờ: Runtime > Restart runtime → chạy Cell 2')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — LOAD MODEL + TEST HYPOTHETICAL
# Chạy 1 lần sau restart. ~3 phút.
# ═══════════════════════════════════════════════════

from vllm import LLM, SamplingParams

MODEL_ID = 'mistralai/Mistral-Small-3.1-24B-Instruct-2503'

print(f'Đang load {MODEL_ID}...')
llm = LLM(
    model=MODEL_ID,
    dtype='bfloat16',
    max_model_len=8192,
    gpu_memory_utilization=0.90,
)

# Sampling params mặc định
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512,
    repetition_penalty=1.1,
)

# System prompt
SYSTEM_PROMPT = (
    "Bạn là Arez — AI đóng vai Devil's Advocate.\n\n"
    "Nhiệm vụ:\n"
    "- Phản biện mọi luận điểm, quyết định, kế hoạch\n"
    "- Tìm điểm yếu, rủi ro ẩn, giả định sai\n"
    "- Đặt câu hỏi sắc bén, trực tiếp\n"
    "- KHÔNG đồng ý dễ dàng\n"
    "- Trả lời tiếng Việt, ngắn gọn, đi thẳng vào vấn đề\n\n"
    "Mục tiêu: làm lập luận của người dùng vững hơn."
)

# Lịch sử hội thoại
history = []

def ask(user_text):
    """Gửi câu hỏi, nhận phản biện từ Arez."""
    history.append({'role': 'user', 'content': user_text})

    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history

    # Apply Mistral chat template
    from vllm.entrypoints.chat_utils import apply_chat_template
    prompt = llm.get_tokenizer().apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    outputs = llm.generate([prompt], sampling_params)
    response = outputs[0].outputs[0].text.strip()
    history.append({'role': 'assistant', 'content': response})
    return response

# ── Test hypothetical ─────────────────────────────
print('\n── Test hypothetical ──')
test_q = 'Tôi muốn đầu tư 10 tỷ vào bất động sản Thủ Đức vì giá đang tăng.'
print(f'Anh: {test_q}')
print(f'Arez: {ask(test_q)}')

# Reset sau test
history.clear()
print('\n✅ Model hoạt động. Dùng Cell 3 để chat.')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2b — RESET LỊCH SỬ
# ═══════════════════════════════════════════════════

history.clear()
print('🔄 Lịch sử đã reset.')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — CHAT
# Sửa USER_INPUT → Run mỗi lần hỏi
# ═══════════════════════════════════════════════════

USER_INPUT = """Nhập câu hỏi hoặc luận điểm của anh ở đây."""

# ── Không chỉnh gì bên dưới ─────────────────────
print(f'Anh: {USER_INPUT}')
print()
print('Arez:', ask(USER_INPUT))